# Graph Representation Learning
## DeepWalk · Node2Vec · GCN

Simple implementations on the same small graph.

---
## The Graph We Use Throughout

```
A ----- B
|       |
C ----- D
```

- 4 nodes: A, B, C, D
- A connects to B and C
- B connects to A and D
- C connects to A and D
- D connects to B and C

In [5]:
# Our graph as a dictionary
# Each key = node, value = list of its neighbors
graph = {
    'A': ['B', 'C'],
    'B': ['A', 'D'],
    'C': ['A', 'D'],
    'D': ['B', 'C']
}

print("Graph adjacency list:")
for node, neighbors in graph.items():
    print(f"  {node} → {neighbors}")

Graph adjacency list:
  A → ['B', 'C']
  B → ['A', 'D']
  C → ['A', 'D']
  D → ['B', 'C']


---
## Part 1 — Random Walk

A random walk starts at a node and randomly moves to one of its neighbors.

This is the foundation of both DeepWalk and Node2Vec.

In [6]:
import random

def random_walk(graph, start_node, walk_length):
    walk = [start_node]
    current = start_node

    for _ in range(walk_length - 1):
        neighbors = graph[current]
        current = random.choice(neighbors)  # pick any neighbor randomly
        walk.append(current)

    return walk

# Example: one random walk starting from A
walk = random_walk(graph, start_node='A', walk_length=6)
print("Random walk:", ' → '.join(walk))

Random walk: A → B → A → B → A → C


---
## Part 2 — DeepWalk

DeepWalk generates many random walks, then feeds them into Word2Vec.

- Each walk = a sentence
- Each node = a word
- Word2Vec learns: nodes appearing together → similar embeddings

In [7]:
# Generate many random walks (one from each node, repeated)
all_walks = []

num_walks   = 10   # how many times we start from each node
walk_length = 5    # how long each walk is

for _ in range(num_walks):
    for node in graph:
        walk = random_walk(graph, node, walk_length)
        all_walks.append(walk)

print(f"Total walks generated: {len(all_walks)}")
print("\nFirst 5 walks:")
for w in all_walks[:5]:
    print(' →'.join(w))

Total walks generated: 40

First 5 walks:
A →B →D →C →D
B →D →B →D →B
C →D →B →D →C
D →B →D →C →A
A →C →D →C →D


In [8]:
from gensim.models import Word2Vec

# Train Word2Vec on the walks
# sg=1 means Skip-Gram (which is what DeepWalk uses)
model = Word2Vec(
    sentences   = all_walks,
    vector_size = 4,     # embedding size (small for demo)
    window      = 2,     # context window
    min_count   = 1,
    sg          = 1,
    epochs      = 100
)

print("DeepWalk Embeddings:")
for node in ['A', 'B', 'C', 'D']:
    vec = model.wv[node].round(3)
    print(f"  {node}: {vec}")

DeepWalk Embeddings:
  A: [-0.285 -0.127  0.301  0.085]
  B: [-0.244  0.14  -0.026  0.057]
  C: [-0.167 -0.018  0.235  0.345]
  D: [-0.376 -0.206  0.265  0.339]


In [9]:
# Check similarity between nodes
# Connected nodes should have higher similarity
print("Similarity scores (higher = more similar):")
print(f"  A and B (connected) : {model.wv.similarity('A', 'B'):.3f}")
print(f"  A and D (not direct): {model.wv.similarity('A', 'D'):.3f}")

Similarity scores (higher = more similar):
  A and B (connected) : 0.383
  A and D (not direct): 0.901


this is because word2vec focuses on structural similarity not on edge connectivity if A and D appear frequently together in generated random walks then that is the reason they are having high random walks.

---
## Part 3 — Node2Vec

Node2Vec is DeepWalk with **smarter walks**.

Two parameters control the walk:
- `p` — how likely to go **back** to previous node
- `q` — how likely to **explore** new nodes

Everything else (Word2Vec training) is identical to DeepWalk.

> `p=1, q=1` → same as DeepWalk

In [10]:
def node2vec_walk(graph, start_node, walk_length, p=1.0, q=1.0):
    walk = [start_node]
    walk.append(random.choice(graph[start_node]))  # first step is random

    for _ in range(walk_length - 2):
        current  = walk[-1]
        previous = walk[-2]
        neighbors = graph[current]

        weights = []
        for neighbor in neighbors:
            if neighbor == previous:
                weights.append(1 / p)   # going back
            elif neighbor in graph[previous]:
                weights.append(1.0)     # common neighbor
            else:
                weights.append(1 / q)  # exploring new area

        # Convert weights to probabilities
        total = sum(weights)
        probs = [w / total for w in weights]

        # Weighted random choice
        r, cumulative = random.random(), 0
        chosen = neighbors[-1]
        for i, prob in enumerate(probs):
            cumulative += prob
            if r <= cumulative:
                chosen = neighbors[i]
                break

        walk.append(chosen)

    return walk

# p=1, q=0.5 → explore more (DFS-like)
walk = node2vec_walk(graph, 'A', walk_length=6, p=1.0, q=0.5)
print("Node2Vec walk (p=1, q=0.5):", ' → '.join(walk))

Node2Vec walk (p=1, q=0.5): A → C → D → B → A → B


In [11]:
# Generate Node2Vec walks and train
n2v_walks = []
for _ in range(10):
    for node in graph:
        walk = node2vec_walk(graph, node, walk_length=5, p=1.0, q=0.5)
        n2v_walks.append(walk)

n2v_model = Word2Vec(
    sentences   = n2v_walks,
    vector_size = 4,
    window      = 2,
    min_count   = 1,
    sg          = 1,
    epochs      = 100
)

print("Node2Vec Embeddings:")
for node in ['A', 'B', 'C', 'D']:
    vec = n2v_model.wv[node].round(3)
    print(f"  {node}: {vec}")

Node2Vec Embeddings:
  A: [-0.148 -0.025  0.231  0.318]
  B: [-0.382 -0.215  0.275  0.326]
  C: [-0.265 -0.129  0.29   0.057]
  D: [-0.234  0.131 -0.028  0.036]


---
## Part 4 — Manual Training Step (What Happens Inside)

This shows **one step** of what Word2Vec is doing internally:

1. Take a training pair (A, B)
2. Compute dot product score
3. Apply Softmax → get probability
4. Compute Loss
5. Update embeddings

In [12]:
import numpy as np

# Starting embeddings (random)
embeddings = {
    'A': np.array([0.2, 0.1]),
    'B': np.array([0.3, 0.5]),
    'C': np.array([0.1, 0.7]),
    'D': np.array([0.4, 0.2]),
}
nodes = ['A', 'B', 'C', 'D']
lr = 0.1   # learning rate

print("Training pair: (A, B)  ← A appeared near B in a walk")
print()

# Step 1: Dot product scores from A to every node
scores = {n: np.dot(embeddings['A'], embeddings[n]) for n in nodes}
print("Step 1 — Dot Product Scores:")
for n, s in scores.items():
    print(f"  score(A, {n}) = {s:.4f}")

Training pair: (A, B)  ← A appeared near B in a walk

Step 1 — Dot Product Scores:
  score(A, A) = 0.0500
  score(A, B) = 0.1100
  score(A, C) = 0.0900
  score(A, D) = 0.1000


In [13]:
# Step 2: Softmax — turn scores into probabilities
exp_scores = {n: np.exp(s) for n, s in scores.items()}
total = sum(exp_scores.values())
probs = {n: e / total for n, e in exp_scores.items()}

print("Step 2 — Softmax Probabilities:")
for n, p in probs.items():
    print(f"  P({n}|A) = {p:.4f}")

Step 2 — Softmax Probabilities:
  P(A|A) = 0.2407
  P(B|A) = 0.2556
  P(C|A) = 0.2506
  P(D|A) = 0.2531


In [14]:
# Step 3: Cross Entropy Loss  (we want P(B|A) to be high)
loss = -np.log(probs['B'])
print(f"Step 3 — Loss = -log(P(B|A)) = -log({probs['B']:.4f}) = {loss:.4f}")
print("(Lower loss = better. We want this to decrease over time.)")

Step 3 — Loss = -log(P(B|A)) = -log(0.2556) = 1.3641
(Lower loss = better. We want this to decrease over time.)


In [15]:
# Step 4: Update embeddings (gradient descent)
old_A = embeddings['A'].copy()
error = probs['B'] - 1
grad  = error * embeddings['B']
embeddings['A'] = embeddings['A'] - lr * grad

print("Step 4 — Update Embedding of A:")
print(f"  Before: {old_A}")
print(f"  After : {embeddings['A'].round(4)}")
print()
print("After many such steps, A and B embeddings become closer together.")

Step 4 — Update Embedding of A:
  Before: [0.2 0.1]
  After : [0.2223 0.1372]

After many such steps, A and B embeddings become closer together.


---
## Part 5 — GCN (Graph Convolutional Network)

GCN doesn't use random walks.
It uses **node features** directly.

For every node:
1. Collect features of itself and its neighbors
2. Multiply by a weight matrix
3. Apply ReLU

**Formula:** `H = ReLU( A @ X @ W )`

| Symbol | Meaning |
|--------|---------|
| `A`    | Adjacency matrix (with self-loops) |
| `X`    | Node feature matrix |
| `W`    | Weight matrix (learned) |

In [16]:
import numpy as np

# Adjacency Matrix WITH self-loops (each node also connects to itself)
#        A  B  C  D
A_mat = np.array([
    [1, 1, 1, 0],   # A: self, B, C
    [1, 1, 0, 1],   # B: A, self, D
    [1, 0, 1, 1],   # C: A, self, D
    [0, 1, 1, 1],   # D: B, C, self
])

# Feature Matrix — each row = one node's feature vector
#        f1  f2
X = np.array([
    [1, 0],   # A
    [1, 1],   # B
    [0, 1],   # C
    [1, 0],   # D
])

print("Adjacency Matrix (with self-loops):")
print(A_mat)
print("\nFeature Matrix X:")
print(X)

Adjacency Matrix (with self-loops):
[[1 1 1 0]
 [1 1 0 1]
 [1 0 1 1]
 [0 1 1 1]]

Feature Matrix X:
[[1 0]
 [1 1]
 [0 1]
 [1 0]]


In [17]:
# Step 1: A @ X  →  aggregate neighbor features for each node
AX = A_mat @ X

print("Step 1 — A @ X (each node collects its neighbors' features):")
print(AX)
print()
nodes = ['A', 'B', 'C', 'D']
for i, node in enumerate(nodes):
    print(f"  {node}: {AX[i]}  ← sum of features of {node} and its neighbors")

Step 1 — A @ X (each node collects its neighbors' features):
[[2 2]
 [3 1]
 [2 1]
 [2 2]]

  A: [2 2]  ← sum of features of A and its neighbors
  B: [3 1]  ← sum of features of B and its neighbors
  C: [2 1]  ← sum of features of C and its neighbors
  D: [2 2]  ← sum of features of D and its neighbors


In [18]:
# Step 2: (A @ X) @ W  →  apply learned weights
np.random.seed(42)
W = np.array([
    [0.5, 0.2],
    [0.3, 0.7]
])

AXW = AX @ W
print("Step 2 — (A @ X) @ W (linear transformation):")
print(AXW)

Step 2 — (A @ X) @ W (linear transformation):
[[1.6 1.8]
 [1.8 1.3]
 [1.3 1.1]
 [1.6 1.8]]


In [19]:
# Step 3: ReLU  →  set all negatives to 0
H = np.maximum(0, AXW)

print("Step 3 — ReLU → Final GCN Embeddings:")
print(H)
print()
print("Final Node Embeddings:")
for i, node in enumerate(nodes):
    print(f"  {node}: {H[i]}")

Step 3 — ReLU → Final GCN Embeddings:
[[1.6 1.8]
 [1.8 1.3]
 [1.3 1.1]
 [1.6 1.8]]

Final Node Embeddings:
  A: [1.6 1.8]
  B: [1.8 1.3]
  C: [1.3 1.1]
  D: [1.6 1.8]


---
## Summary — Side by Side

| | DeepWalk | Node2Vec | GCN |
|--|--|--|--|
| Uses random walks | ✅ | ✅ | ❌ |
| Uses node features | ❌ | ❌ | ✅ |
| p, q parameters | ❌ | ✅ | ❌ |
| Output | Node embeddings | Node embeddings | Node embeddings |

> All three methods produce node embeddings — just through different approaches.

In [20]:
print("Quick recap:")
print()
print("DeepWalk : Graph → Uniform Random Walks → Word2Vec → Embeddings")
print("Node2Vec : Graph → Biased Walks (p,q)   → Word2Vec → Embeddings")
print("GCN      : Graph + Features → A @ X @ W → ReLU    → Embeddings")
print()
print("GNN is the family.  GCN is one member of that family.")
print("DeepWalk = Node2Vec with p=1, q=1")

Quick recap:

DeepWalk : Graph → Uniform Random Walks → Word2Vec → Embeddings
Node2Vec : Graph → Biased Walks (p,q)   → Word2Vec → Embeddings
GCN      : Graph + Features → A @ X @ W → ReLU    → Embeddings

GNN is the family.  GCN is one member of that family.
DeepWalk = Node2Vec with p=1, q=1
